# sync

> Push the local archive checkout to the box: `reconcile-push`, the monthly one-liner

In [ ]:
#| default_exp sync

In [ ]:
#| hide
from nbdev.showdoc import *

The laptop's `~/reconcile-archive` checkout is the source of truth; the box's
viewer reads a pushed copy at `/srv/reconcile-archive`. The `--chmod=D2750,F640`
policy is what makes the copy readable by the box's `reconcile` service user
(group-read, setgid dirs) and by nobody else — baking it in here is the point
of the command. The app is stateless: after a push the new months are live,
no restart needed.

## `push_cmd`

In [ ]:
#| export
from pathlib import Path

def push_cmd(
    src:str="~/reconcile-archive",  # archive checkout (source of truth)
    dest:str="doyu@box.ninjalabo.ai:/srv/reconcile-archive",  # rsync target on the box
    dry_run:bool=False,             # add rsync --dry-run
)->list:                            # rsync argv
    "Build the rsync argv: archive push with the box's group-read/setgid policy baked in."
    src = str(Path(src).expanduser())
    # --no-group: -a would preserve the laptop's group (doyu), locking out the
    # box's reconcile service user; the destination's setgid dir sets the group
    return ["rsync", "-a", "--no-group", "--delete", "--chmod=D2750,F640",
            *(["--dry-run"] if dry_run else []),
            f"{src.rstrip('/')}/", f"{dest.rstrip('/')}/"]

In [ ]:
cmd = push_cmd()
assert cmd[0] == "rsync"
assert "-a" in cmd and "--delete" in cmd
# the box's permission policy is baked in, not remembered by humans
assert "--chmod=D2750,F640" in cmd
# -a implies -g, which as doyu re-groups everything to doyu and locks out the
# box's reconcile service user (the 500 we shipped on day one) — never preserve
# the laptop's group; the setgid dir assigns group reconcile instead
assert "--no-group" in cmd
# trailing slashes on both ends: sync the CONTENTS of src into dest, no nested dir
assert cmd[-2].endswith("/") and not cmd[-2].endswith("//")
assert cmd[-1] == "doyu@box.ninjalabo.ai:/srv/reconcile-archive/"
# ~ is expanded so subprocess (no shell) can use the path
assert "~" not in cmd[-2]
# dry_run toggles rsync's own flag
assert "--dry-run" not in cmd
assert "--dry-run" in push_cmd(dry_run=True)
# overrides pass through
c2 = push_cmd(src="/tmp/a", dest="host:/x")
assert c2[-2] == "/tmp/a/" and c2[-1] == "host:/x/"

## `reconcile_push`

In [ ]:
#| export
import subprocess, sys
from fastcore.script import call_parse

@call_parse
def reconcile_push(
    src:str="~/reconcile-archive",  # archive checkout (source of truth)
    dest:str="doyu@box.ninjalabo.ai:/srv/reconcile-archive",  # rsync target on the box
    dry_run:bool=False,             # preview: rsync --dry-run
):
    "Push the archive to the box (the viewer reads it live; no restart needed)"
    cmd = push_cmd(src, dest, dry_run)
    print(" ".join(cmd), file=sys.stderr)
    raise SystemExit(subprocess.run(cmd).returncode)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()